In [ ]:
# Lataa alta tarvittavat kirjastot
# %pip install pandas matplotlib plotly dash

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
from plotly.subplots import make_subplots

## Projektissa käytettävä datasetti

#### Tietoa datasetistä
Datasetti on peräisin [Kagglesta](https://www.kaggle.com/datasets/whisperingkahuna/premier-league-2324-team-and-player-insights/data), ja se tarjoaa kattavan tilastot jalkapallon Valioliigan kaudesta 2023/24, sisältäen dataa joukkueiden ja pelaajien suorituksista kaikilta ottelukierroksilta. Yhteensä datasetti koostuu yli 50 CSV-tiedostosta, joista jokainen keskittyy pelin tiettyihin osa-alueisiin, kuten joukkueiden suorituksiin, pelaajamittareihin, ottelutietoihin ja sarjataulukoihin.

#### Datasetin sisältö
Datasetti sisältää:
- **Joukkueiden suoritusmittarit**: Tarkat syötöt, päästetyt maalit, katkot jne.
- **Pelaajien suoritusmittarit**: Maaliodottama (xG), syötöt, voitetut taklaukset jne.
- **Ottelukohtaiset tiedot**: Tehdyt maalit, pallonhallintaprosentit, annetut kortit jne.
- **Sarjataulukot**: Sijoitukset, koti/vieras-suoritukset, ja edistyneet mittarit kuten xG ja xA.

#### Tiedostojen yksityiskohdat
Jokainen CSV-tiedosto sisältää tiettyyn aiheeseen liittyviä sarakkeita. Esimerkiksi:
- `player_top_scorers.csv`: Sijoitus, Pelaaja, Joukkue, Maalit, Maalit Rangaistuspotkuista, Minuutit, Ottelut, Kansalaisuus.
- `accurate_pass_team.csv`: Sijoitus, Joukkue, Onnistuneet syötöt per ottelu, Syöttöjen onnistumisprosentti (%), Ottelut, Kansalaisuus.
- `player_expected_goals.csv`: Sijoitus, Pelaaja, Joukkue, Maaliodottama (xG), Todelliset maalit, Minuutit, Ottelut.

TODO: Tarkka erittely datasetin sisältämistä tiedostoista ja niiden sisältämistä sarakkeista

## Datan lukeminen ja siivoaminen 

Kaggle tarjoaa useita vaihtoehtoisia tapoja datan tallentamiseen ja lukemiseen. Tämän projektin kohdalla päädyttiin viemään data GitHubiin, jotta vertaisarviointi onnistuu helposti (ts. datan lukeminen ei vaadi vertaisarvioijalta erillisiä toimenpiteitä).
Data luetaan Pandas DataFrameihin GitHubista analysointia varten.

Rakennetaan yksinkertainen funktio tiedostojen GitHubista lukemista varten:

In [ ]:
def readFile (filename, inMainDir = True):
    filename = filename.strip()
    baseURL = "https://raw.githubusercontent.com/mikaelkankaanpaa/datatie2025_mk/main/project/data/"
    if inMainDir:
        url = f"{baseURL}Premleg_23_24/{filename}"
    else:
        url = f"{baseURL}{filename}"
    try:
        df = pd.read_csv(url)
    except Exception as e:
        print(f"Dataa ei voitu lukea. Tarkista tiedoston nimi!\nVirheilmoitus: {e}")
        return None
    return df

Testataan lukea yksi datasetin tiedostoista (pelaajien tehdyt maalit; `player_top_scorers.csv`) dataFrameen, ja muodostetaan yleiskäsitys tiedoston sisällöstä:

In [ ]:
# topScorersDF = readFile("testi_vaara_nimi.csv") 
topScorersDF = readFile("player_top_scorers.csv")

if topScorersDF is not None:
    # dataFramen perustiedot
    display(topScorersDF.info())
    # ensimmäiset 5 riviä
    display(topScorersDF.head())
    # viimeiset 5 riviä
    display(topScorersDF.tail())

Datassa ei näytä olevan lainkaan NULL-arvoja, ja datatyypitkin vaikuttavat sopivilta (ks. `info()`:n printout). Siirrytään muokkaamaan ja jalostamaan dataa.

## Datan muokkaaminen & jalostaminen 

Datassa ei ole omaa saraketta pelitilannemaaleille (ts. ei-pilkkumaaleille); lisätään se. Uudelleennimetään samalla 'Goals' -> 'Total Goals':

In [ ]:
topScorersDF.rename(columns={'Goals': 'Total Goals'}, inplace=True)

topScorersDF.insert(
    loc = 4, # yhteismaalien ja pilkkujen väliin
    column ='Non-Penalty Goals',
    value = topScorersDF['Total Goals'] - topScorersDF['Penalties']
)

Lisätään myös sarakkeet "Goals per Game" ja "Goals per 60 Min", jotka nimensä mukaisesti kertovat pelikohtaisen maalimäärän (ts. maalimäärä normalisoidaan suhteessa pelattuihin otteluihin/peliaikaan). Käytetään peliajan suhteen yksikkönä maalia/60 min, joka on intuitiivisempi kuin maalia/min. 

In [ ]:
topScorersDF['Goals per Game'] = topScorersDF['Total Goals'] / topScorersDF['Matches']
topScorersDF['Goals per 60 Min'] = topScorersDF['Total Goals'] / (topScorersDF['Minutes'] / 60)

# Testataan viidellä satunnaisella pelaajalla:
display(topScorersDF.sample(5).sort_values(by='Goals per Game', ascending=False))

## Datan visualisointi

Tässä osiossa testataan 3 eri tapaa visualisoida Pandas-dataa (edellä esitelty maalintekijät-dataframe) Pythonilla: 
- simppeli Matplotlib; 
- vähän edistyneempi Plotly Express, joka mahdollistaa jo ihan kiitettävästi interaktiivisuutta kuvaajaan; 
- tästä vielä monipuolisemmat Plotly Dash-apit, joihin voidaan integroida mm. HTML/CSS-komponentteja erilaisten interaktiivisten komponettien luomiseksi.


### 1. Matplotlib 
Visualisoidaan top 10 maalintekijää dataFramen ja matplotlib-kirjaston tarjoamien visualisointityökalujen avulla. 
Lisätään 'peruskuvaajaan' pelaajakohtaiset rankkarimaalit, sekä selite jokaisen palkin perään, joka ilmaisee tarkan pelaajakohtaisen kokonaismaalimäärän, jotta kuvaaja on informatiivisempi:

(Matplotlibin värit: https://matplotlib.org/stable/gallery/color/named_colors.html)

In [ ]:
plt.figure(figsize=(10, 5))

# palkit 
barsGoals = (plt.barh(
    topScorersDF['Player'][:10], 
    topScorersDF['Total Goals'][:10], 
    label='Pelitilannemaalit',
    color='darkslateblue')
) 
barsPenalties = (plt.barh(
    topScorersDF['Player'][:10], topScorersDF['Penalties'][:10], 
    left=topScorersDF['Total Goals'][:10] - topScorersDF['Penalties'][:10], 
    label='Rangaistuspotkumaalit',
    color='mediumspringgreen')
)
# labelit ja otsikko
plt.xlabel("Maalit")
plt.ylabel("Pelaaja")
plt.title("Top 10 Maalintekijät Valioliigassa kaudella 2023/24")
plt.gca().invert_yaxis()

# selite maalien kokonaismäärästä 
for bar in barsGoals:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, 
             f'{int(bar.get_width())}', va='center')

plt.legend()
plt.show()


### 2. Interaktiivinen kuvaaja Plotly Expressillä

Kokeillaan samalla datalla rakentaa interaktiivinen pylväskaavio [Plotly](https://plotly.com/python)-kirjaston avulla.
- (Plotly pylväskaaviot -tutoriaali: https://plotly.com/python/bar-charts/)
- (Plotly Express pylväskaavioiden dokumentaatio: https://plotly.com/python-api-reference/generated/plotly.express.bar.html#plotly.express.bar)

In [ ]:
# Muutetaan ("sulatetaan") dataFrame wide formatista long formatiin, 
# jotta se sopii Plotly Expressin käyttöön ja uudelleennimetään sarakkeet suomeksi kuvaajaa varten
meltedTop10DF = topScorersDF[:10].rename(
    columns={
        'Player': 'Pelaaja',
        'Non-Penalty Goals': 'Pelitilannemaalit', 
        'Penalties': 'Rangaistuspotkumaalit'
    }
).melt(
    id_vars='Pelaaja', 
    value_vars=['Pelitilannemaalit', 'Rangaistuspotkumaalit'], 
    var_name='Maalityyppi', 
    value_name='Määrä'
)

fig2 = px.bar(
    meltedTop10DF,
    x='Määrä',
    y='Pelaaja',
    color='Maalityyppi',
    orientation='h',
    title="Top 10 Maalintekijät Valioliigassa kaudella 2023/24",
    color_discrete_map={'Pelitilannemaalit': 'darkslateblue', 'Rangaistuspotkumaalit': 'mediumspringgreen'}
)
fig2.update_layout(
    barmode='stack',  # pinottu pylväsdiagrammi (toimii ilmankin)
    yaxis=dict(autorange="reversed"),  # käänteinen y:n järjestys
    autosize=False,
    width=1200
)
fig2.show()

Kuvaajassa voi nyt esim. valita näkyviin haluamansa maalityypin (klikkaamalla maalityyppiä legendissä -> päälle/pois), hiiri palkin päälle viemällä selite näkyviin jne.

### 3. Dynaaminen kuvaaja Dashilla (+ Plotly Express)

Lisätään Plotly Expressillä toteutettuun kuvaajaan liukusäädin, jolla voi säätää näytettävien maalintekijöiden määrää. Ts. kuvaaja näyttää dynaamisesti sliderin valinnan mukaan 5-20 parasta maalintekijää. 

(Dash Python User Guide: https://dash.plotly.com/)

<div class="alert alert-block alert-danger">

<b>
Tätä sais vielä siistiä/parannella, mod 5 palautuksessa vielä aika MVP. Lisää ideoita listaan, jos tulee mieleen :) <br>
TODO:<br>
- Pelaajamäärän mukaan dynaamisesti muuttuva kuvaajan/plotarean korkeus (atm esim. 20 pelaajan nimet ei mahdu)<br>
- Sliderin ulkoasu</b>
</div>



In [ ]:
app = Dash(__name__)

# Apin html-layout   
app.layout = html.Div([
    html.Div(
        children=[
            dcc.Slider(
                id='top-n-slider',
                min=5,
                max=20,
                step=1,
                value=10,  # default
                marks={i: str(i) for i in range(5, 21)},
                # tooltip={"placement": "bottom", "always_visible": True},
            )
        ],
        # CSS-tyylit
        style={
            'backgroundColor': 'white',  
            'padding': '40px',  
            'margin': 'left auto',  
            'width': '1120px',
        }
    ),
    dcc.Graph(id='top-scorers-plot')
])

# Callback plotin päivittämiseen
@app.callback(
    Output('top-scorers-plot', 'figure'),
    Input('top-n-slider', 'value')
)
def update_plot(n):
    # Filtteröidään top n pelaajaa ja uudelleennimetään sarakkeet suomeksi
    filteredDF = topScorersDF[:n].rename(
        columns={
            'Player': 'Pelaaja',
            'Non-Penalty Goals': 'Pelitilannemaalit',
            'Penalties': 'Rangaistuspotkumaalit'
        }
    ).melt(
        id_vars='Pelaaja',
        value_vars=['Pelitilannemaalit', 'Rangaistuspotkumaalit'],
        var_name='Maalityyppi',
        value_name='Määrä'
    )
    # Päivitetty kuvaaja
    fig = px.bar(
        filteredDF,
        x='Määrä',
        y='Pelaaja',
        color='Maalityyppi',
        orientation='h',
        title=f"Top {n} Maalintekijät Valioliigassa 2023/24",
        color_discrete_map={'Pelitilannemaalit': 'darkslateblue', 'Rangaistuspotkumaalit': 'mediumspringgreen'}
    )
    fig.update_layout(
        barmode='stack',
        yaxis=dict(autorange="reversed"),
        autosize=False,
        width=1200
    )
    return fig

# Ajetaan Dash-appi
if __name__ == '__main__':
    app.run(debug=True)

## Analyysi
TODO


In [ ]:

top10absDF = topScorersDF.sort_values(by='Total Goals', ascending=False)[:10]
top10per60DF = topScorersDF.sort_values(by='Goals per 60 Min', ascending=False)[:10]


fig4 = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Top 10 Maalimäärä", "Top 10 Maalit per 60 Minuuttia")
)

fig4.add_trace(
    px.bar(
        top10absDF,
        x='Total Goals',
        y='Player',
        orientation='h',
        title="",
        labels={'Total Goals': 'Maalit yhteensä', 'Player': 'Pelaaja'}
    ).data[0],
    row=1, col=1
)
fig4.add_trace(
    px.bar(
        top10per60DF,
        x='Goals per 60 Min',
        y='Player',
        orientation='h',
        title="",
        labels={'Goals per 60 Minutes': 'Maalit per 60 Minuuttia', 'Player': 'Pelaaja'}
    ).data[0],
    row=1, col=2
)

# Update layout
fig4.update_layout(
    title_text="Top 10 Maalintekijät Valioliigassa kaudella 2023/24",
    autosize=False,
    width=1400,
    height=600,
    showlegend=True,
    # yaxis=dict(autorange="reversed")
)

# Show the plot
fig4.show()